# Milestone 2: Data Quality dan Profiling (Time Series)

Lanjutan dari `01_data_profiling_timeseries.ipynb` pada dataset **PEMS-BAY**. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar pada data deret waktu lalu lintas.

Pembagian peran:
- **DuckDB** untuk profiling dan validasi (`SUMMARIZE`, query SQL aturan kualitas).
- **Polars (lazy)** untuk transformasi dan penyimpanan hasil ke `data/processed/` lewat `sink_parquet` (streaming, tidak memuat semua data ke RAM).

In [1]:
from pathlib import Path

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'pems-bay'
DATA_GLOB = (RAW_DIR / '*.parquet').as_posix()

import duckdb
import polars as pl

OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'pems_bay_clean.parquet'

if len(list(RAW_DIR.glob('*.parquet'))) < 3:
    raise FileNotFoundError(
        f'Dataset tidak ditemukan di {RAW_DIR}. '
        'Jalankan 01_data_profiling_timeseries.ipynb terlebih dahulu untuk mengunduh data.'
    )

lf = pl.scan_parquet(DATA_GLOB, include_file_paths='sumber')
skema = lf.collect_schema()
KOLOM_D0 = [c for c in skema if c.endswith('_d0')]
KOLOM_D1 = [c for c in skema if c.endswith('_d1')]
KOLOM_ASLI = [c for c in skema if c != 'sumber']

jumlah_baris = lf.select(pl.len()).collect(engine='streaming').item()
print(f'Ukuran awal: {jumlah_baris:,} baris x {len(KOLOM_ASLI)} kolom')
print(f'Kolom kecepatan (d0): {len(KOLOM_D0)}, kolom waktu-hari (d1): {len(KOLOM_D1)}')
lf.select('node_id', 't0_timestamp', 'x_t+0_d0', 'x_t+0_d1', 'y_t+1_d0').head().collect(engine='streaming')

Ukuran awal: 16,930,225 baris x 50 kolom
Kolom kecepatan (d0): 24, kolom waktu-hari (d1): 24


node_id,t0_timestamp,x_t+0_d0,x_t+0_d1,y_t+1_d0
i64,str,f64,f64,f64
0,"""2017-05-25T18:45:00""",73.1,0.78125,70.9
1,"""2017-05-25T18:45:00""",22.5,0.78125,24.1
2,"""2017-05-25T18:45:00""",56.7,0.78125,54.4
3,"""2017-05-25T18:45:00""",32.3,0.78125,32.6
4,"""2017-05-25T18:45:00""",58.0,0.78125,62.5


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik (tipe data, null%, min/max, distinct count) dalam satu perintah. Dataset memiliki 50 kolom dengan struktur berulang, sehingga `SUMMARIZE` dijalankan pada kolom kunci dan beberapa horizon prediksi.

In [2]:
connection = duckdb.connect()

profil = connection.sql(f'''
    SUMMARIZE SELECT node_id, t0_timestamp,
        "x_t-11_d0", "x_t+0_d0", "x_t+0_d1", "y_t+1_d0", "y_t+6_d0", "y_t+12_d0", "y_t+12_d1"
    FROM read_parquet('{DATA_GLOB}')
''').pl()
profil.select('column_name', 'column_type', 'min', 'max', 'approx_unique', 'avg', 'std', 'null_percentage')

column_name,column_type,min,max,approx_unique,avg,std,null_percentage
str,str,str,str,i64,str,str,"decimal[9,2]"
"""node_id""","""BIGINT""","""0""","""324""",325,"""162.0""","""93.81897739742516""",0.00
"""t0_timestamp""","""VARCHAR""","""2017-01-01T00:55:00""","""2017-06-30T22:55:00""",52707,null,null,0.00
"""x_t-11_d0""","""DOUBLE""","""0.0""","""85.1""",798,"""62.61772499183997""","""9.595799942893633""",0.00
"""x_t+0_d0""","""DOUBLE""","""0.0""","""85.1""",798,"""62.61742577549769""","""9.5958238940905""",0.00
"""x_t+0_d1""","""DOUBLE""","""0.0""","""0.9965277777777778""",286,"""0.4983462402711381""","""0.28853146439422345""",0.00
"""y_t+1_d0""","""DOUBLE""","""0.0""","""85.1""",798,"""62.61740679760462""","""9.595818415003878""",0.00
"""y_t+6_d0""","""DOUBLE""","""0.0""","""85.1""",798,"""62.617296869945505""","""9.595786954643343""",0.00
"""y_t+12_d0""","""DOUBLE""","""0.0""","""85.1""",798,"""62.6171842547853""","""9.595760446356216""",0.00
"""y_t+12_d1""","""DOUBLE""","""0.0""","""0.9965277777777778""",286,"""0.49855820093124""","""0.28854535488857747""",0.00


## 2. Deteksi Missing Values

Pada data deret waktu, missing values muncul dalam dua bentuk:
1. **`null`** yang dihitung `null_count`.
2. **Nilai kecepatan `0`**: sensor PEMS-BAY mencatat `0` ketika pembacaan hilang, bukan kendaraan benar-benar berhenti. Bentuk ini tidak terdeteksi `null_count`, sehingga dihitung terpisah.

Selain itu diperiksa **timestamp yang hilang** (celah pada deret 5 menit), karena celah mengubah makna operasi seperti `shift()` dan `rolling_mean()`.

**Alasan:** jika `0` dibiarkan sebagai kecepatan, rata-rata dan tren akan terseret turun secara palsu, dan model peramalan belajar dari pembacaan yang sebenarnya tidak ada.

In [3]:
total_null = lf.select(pl.sum_horizontal(pl.col(KOLOM_ASLI).null_count())).sum().collect(engine='streaming').item()
print(f'Total null seluruh kolom: {total_null:,}')

nol_per_kolom = lf.select([(pl.col(c) == 0).sum().alias(c) for c in KOLOM_D0]).collect(engine='streaming').row(0, named=True)
total_nol = sum(nol_per_kolom.values())
n_nilai_d0 = jumlah_baris * len(KOLOM_D0)
print(f'Nilai kecepatan 0       : {total_nol:,} dari {n_nilai_d0:,} ({total_nol / n_nilai_d0 * 100:.4f}%)')

baris_ada_nol = lf.filter(pl.any_horizontal([pl.col(c) == 0 for c in KOLOM_D0])).select(pl.len()).collect(engine='streaming').item()
print(f'Baris yang memuat >=1 nol: {baris_ada_nol:,} ({baris_ada_nol / jumlah_baris * 100:.3f}%)')

nol_per_sensor = (
    lf.filter(pl.col('x_t+0_d0') == 0).group_by('node_id').agg(pl.len().alias('jumlah_nol_saat_ini'))
    .sort('jumlah_nol_saat_ini', descending=True).head(5).collect(engine='streaming')
)
print()
print('Sensor dengan nol terbanyak pada x_t+0_d0:')
print(nol_per_sensor)

per_timestamp = (
    lf.select(pl.col('t0_timestamp').str.to_datetime('%Y-%m-%dT%H:%M:%S').alias('t0'))
    .unique().sort('t0').collect(engine='streaming')
)
celah = per_timestamp.with_columns(pl.col('t0').diff().alias('selisih')).filter(pl.col('selisih') > pl.duration(minutes=5))
print()
print(f'Timestamp unik: {per_timestamp.height:,}; celah > 5 menit: {celah.height}')
celah

Total null seluruh kolom: 0


Nilai kecepatan 0       : 12,504 dari 406,325,400 (0.0031%)


Baris yang memuat >=1 nol: 12,504 (0.074%)

Sensor dengan nol terbanyak pada x_t+0_d0:
shape: (5, 2)
┌─────────┬─────────────────────┐
│ node_id ┆ jumlah_nol_saat_ini │
│ ---     ┆ ---                 │
│ i64     ┆ u32                 │
╞═════════╪═════════════════════╡
│ 304     ┆ 4                   │
│ 255     ┆ 4                   │
│ 244     ┆ 4                   │
│ 240     ┆ 4                   │
│ 237     ┆ 4                   │
└─────────┴─────────────────────┘



Timestamp unik: 52,093; celah > 5 menit: 1


t0,selisih
datetime[μs],duration[μs]
2017-03-12 03:00:00,1h 5m


## 3. Penanganan Missing Values

Strategi yang dipilih:
- **Nilai `0` pada kolom kecepatan diubah menjadi `null`** (bukan baris dibuang), sehingga jendela 12 langkah tiap baris tetap utuh dan perhitungan agregat (`mean`, `rolling_mean`) otomatis mengabaikan pembacaan yang hilang.
- Kolom `n_hilang` ditambahkan sebagai jumlah nilai kecepatan hilang per baris, supaya baris yang kualitasnya rendah mudah disaring pada tahap analisis atau pemodelan.
- **Tidak ada imputasi** (interpolasi atau forward-fill): baris yang berdekatan berbagi nilai yang sama pada jendelanya, sehingga mengisi nilai di satu baris akan membuat baris lain tidak konsisten, dan nilai karangan dapat menyembunyikan gangguan sensor.
- Tidak ada baris yang dibuang karena `null` pada kolom kunci (`node_id`, `t0_timestamp`), karena hasil pemeriksaan di atas menunjukkan jumlahnya nol.

**Alasan:** pada deret waktu, menghapus baris merusak keteraturan interval 5 menit, sedangkan menandai nilai hilang sebagai `null` mempertahankan struktur waktu dan tetap jujur terhadap data asli.

In [4]:
lf_missing_handled = lf.with_columns(
    [pl.when(pl.col(c) == 0).then(None).otherwise(pl.col(c)).alias(c) for c in KOLOM_D0]
).with_columns(
    pl.sum_horizontal([pl.col(c).is_null().cast(pl.Int8) for c in KOLOM_D0]).alias('n_hilang')
).filter(pl.col('node_id').is_not_null() & pl.col('t0_timestamp').is_not_null())

n_setelah_missing = lf_missing_handled.select(pl.len()).collect(engine='streaming').item()
n_null_baru = lf_missing_handled.select(pl.sum_horizontal(pl.col(KOLOM_D0).null_count())).sum().collect(engine='streaming').item()
print(f'Baris sebelum : {jumlah_baris:,}')
print(f'Baris sesudah : {n_setelah_missing:,}')
print(f'Nilai kecepatan yang kini null: {n_null_baru:,} (harus sama dengan jumlah nol: {total_nol:,})')

Baris sebelum : 16,930,225
Baris sesudah : 16,930,225
Nilai kecepatan yang kini null: 12,504 (harus sama dengan jumlah nol: 12,504)


## 4. Deteksi dan Penanganan Outlier

Dua jenis pemeriksaan:
- **Batas domain**: kecepatan harus berada pada 0 s.d. 100 mph (batas fisik wajar jalan raya) dan kolom waktu-hari `d1` pada 0 s.d. 1. Nilai di luar batas adalah data rusak dan dibuang.
- **IQR per sensor** pada `x_t+0_d0`: dihitung per sensor, bukan global, karena tiap ruas jalan punya kecepatan normal berbeda (jalan bebas hambatan lancar vs ruas yang sering padat).

**Alasan:** kecepatan rendah akibat kemacetan adalah fenomena nyata yang justru menjadi sasaran analisis. Karena itu outlier IQR hanya **dilaporkan**, tidak dibuang atau di-clip; hanya pelanggaran batas fisik yang dibuang.

In [5]:
KECEPATAN_MAKS = 100.0

di_luar_kecepatan = lf_missing_handled.filter(
    pl.any_horizontal([(pl.col(c) < 0) | (pl.col(c) > KECEPATAN_MAKS) for c in KOLOM_D0])
).select(pl.len()).collect(engine='streaming').item()
di_luar_d1 = lf_missing_handled.filter(
    pl.any_horizontal([~pl.col(c).is_between(0, 1) for c in KOLOM_D1])
).select(pl.len()).collect(engine='streaming').item()
print(f'Baris dengan kecepatan di luar [0, {KECEPATAN_MAKS:.0f}] : {di_luar_kecepatan:,}')
print(f'Baris dengan d1 di luar [0, 1]          : {di_luar_d1:,}')

kecepatan_saat_ini = lf_missing_handled.select('node_id', pl.col('x_t+0_d0').alias('kecepatan')).drop_nulls().collect(engine='streaming')
batas_sensor = kecepatan_saat_ini.group_by('node_id').agg(
    pl.col('kecepatan').quantile(0.25).alias('q1'),
    pl.col('kecepatan').quantile(0.75).alias('q3'),
).with_columns(
    (pl.col('q3') - pl.col('q1')).alias('iqr'),
).with_columns(
    (pl.col('q1') - 1.5 * pl.col('iqr')).alias('bawah'),
    (pl.col('q3') + 1.5 * pl.col('iqr')).alias('atas'),
)
outlier_iqr = (
    kecepatan_saat_ini.join(batas_sensor, on='node_id')
    .select(
        (pl.col('kecepatan') < pl.col('bawah')).sum().alias('di_bawah'),
        (pl.col('kecepatan') > pl.col('atas')).sum().alias('di_atas'),
        pl.len().alias('total'),
    )
)
print()
print('Outlier IQR per sensor pada x_t+0_d0 (hanya dilaporkan):')
print(outlier_iqr.with_columns((pl.col('di_bawah') / pl.col('total') * 100).round(2).alias('persen_di_bawah')))
print()
print(kecepatan_saat_ini['kecepatan'].describe())

Baris dengan kecepatan di luar [0, 100] : 0
Baris dengan d1 di luar [0, 1]          : 0



Outlier IQR per sensor pada x_t+0_d0 (hanya dilaporkan):
shape: (1, 4)
┌──────────┬─────────┬──────────┬─────────────────┐
│ di_bawah ┆ di_atas ┆ total    ┆ persen_di_bawah │
│ ---      ┆ ---     ┆ ---      ┆ ---             │
│ u32      ┆ u32     ┆ u32      ┆ f64             │
╞══════════╪═════════╪══════════╪═════════════════╡
│ 1572424  ┆ 60729   ┆ 16929704 ┆ 9.29            │
└──────────┴─────────┴──────────┴─────────────────┘



shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 1.6929704e7 │
│ null_count ┆ 0.0         │
│ mean       ┆ 62.619353   │
│ std        ┆ 9.589682    │
│ min        ┆ 3.0         │
│ 25%        ┆ 62.1        │
│ 50%        ┆ 65.3        │
│ 75%        ┆ 67.5        │
│ max        ┆ 85.1        │
└────────────┴─────────────┘


Keputusan penanganan:
- **Kecepatan dan `d1` di luar batas domain**: dibuang (jika ada).
- **Outlier IQR (kecepatan rendah saat macet)**: tidak di-clip dan tidak dibuang, karena merupakan fenomena lalu lintas yang sah dan menjadi objek analisis pada Milestone 3.

**Alasan:** memotong nilai kecepatan rendah akan menghapus kejadian kemacetan, padahal pola kemacetan adalah insight utama analisis lalu lintas.

## 5. Deduplikasi

Kunci bisnis dataset ini adalah pasangan (`node_id`, `t0_timestamp`): setiap sensor hanya boleh punya satu baris untuk satu waktu referensi. Dicek duplikat kunci pada seluruh split (`train`, `val`, `test`) serta duplikat penuh pada seluruh kolom (lewat hash, engine streaming).

**Alasan:**
- Duplikat kunci pada deret waktu menggandakan bobot satu waktu tertentu dan merusak agregasi, `rolling_mean`, dan `shift()`.
- Pengecekan lintas split memastikan tidak ada kebocoran: waktu yang sama tidak boleh muncul di dua split.
- Hash dengan engine streaming dipakai agar tidak memuat seluruh 16,9 juta baris ke RAM.

In [6]:
KUNCI = ['node_id', 't0_timestamp']

n_unik_kunci = lf_missing_handled.select(pl.struct(KUNCI).n_unique()).collect(engine='streaming').item()
n_unik_penuh = (
    lf_missing_handled.select(pl.struct(KOLOM_ASLI).hash().n_unique())
    .collect(engine='streaming').item()
)
split_per_waktu = (
    lf_missing_handled.select('t0_timestamp', pl.col('sumber').str.extract(r'([^/\\]+)\.parquet$', 1).alias('split'))
    .unique().group_by('t0_timestamp').agg(pl.col('split').n_unique().alias('jumlah_split'))
    .filter(pl.col('jumlah_split') > 1).select(pl.len()).collect(engine='streaming').item()
)

print(f'Total baris                         : {n_setelah_missing:,}')
print(f'Baris unik berdasarkan {KUNCI}: {n_unik_kunci:,}')
print(f'Duplikat kunci                      : {n_setelah_missing - n_unik_kunci:,}')
print(f'Duplikat penuh (50 kolom, hash)     : {n_setelah_missing - n_unik_penuh:,}')
print(f'Timestamp yang muncul di >1 split   : {split_per_waktu:,}')

lf_dedup = lf_missing_handled
if n_setelah_missing > n_unik_kunci:
    lf_dedup = lf_dedup.unique(subset=KUNCI, keep='first', maintain_order=False)
    print('Duplikat dibuang dengan unique(subset=KUNCI).')
else:
    print('Tidak ada duplikat, tidak ada baris yang dibuang.')

Total baris                         : 16,930,225
Baris unik berdasarkan ['node_id', 't0_timestamp']: 16,930,225
Duplikat kunci                      : 0
Duplikat penuh (50 kolom, hash)     : 0
Timestamp yang muncul di >1 split   : 0
Tidak ada duplikat, tidak ada baris yang dibuang.


## 6. Validasi Data Dasar

Aturan validasi ditulis sebagai query SQL DuckDB dan dihitung sebagai jumlah pelanggaran:

| Aturan | Keterangan |
|---|---|
| `node_valid` | `node_id` berada pada 0 s.d. 324 (325 sensor) |
| `waktu_valid` | `t0_timestamp` dapat di-parse ke timestamp |
| `kecepatan_valid` | kecepatan `x_t+0_d0` dan `y_t+1_d0` berada pada 0 s.d. 100 mph |
| `d1_valid` | `x_t+0_d1` berada pada 0 s.d. 1 |
| `waktu_konsisten` | `x_t+0_d1` sama dengan jam-menit `t0_timestamp` dibagi 24 jam (toleransi 1e-4) |
| `nol_tersisa` | tidak ada nilai kecepatan `0` pada `x_t+0_d0` (nilai hilang sudah menjadi `null`) |

**Alasan tiap aturan:**
- `node_valid` dan `waktu_valid`: kunci yang rusak membuat baris tidak dapat ditempatkan pada deret waktu sensor.
- `kecepatan_valid` dan `d1_valid`: batas fisik dan batas normalisasi; pelanggaran menandakan data korup.
- `waktu_konsisten`: kolom `d1` diturunkan dari waktu, sehingga selisih dengan `t0_timestamp` menandakan ketidakcocokan antar kolom.
- `nol_tersisa`: bukti bahwa penanganan missing values pada langkah 3 berhasil.
- Validasi memakai SQL DuckDB agar terpisah dari kode pembersihan Polars, sehingga hasil keduanya bisa saling memeriksa.

In [7]:
def hitung_pelanggaran(sumber):
    return connection.sql(f'''
        WITH dasar AS (
            SELECT *, TRY_CAST(t0_timestamp AS TIMESTAMP) AS t0 FROM {sumber}
        )
        SELECT
            COUNT(*) AS total_baris,
            SUM(CASE WHEN node_id NOT BETWEEN 0 AND 324 THEN 1 ELSE 0 END) AS node_valid,
            SUM(CASE WHEN t0 IS NULL THEN 1 ELSE 0 END) AS waktu_valid,
            SUM(CASE WHEN "x_t+0_d0" NOT BETWEEN 0 AND {KECEPATAN_MAKS}
                       OR "y_t+1_d0" NOT BETWEEN 0 AND {KECEPATAN_MAKS} THEN 1 ELSE 0 END) AS kecepatan_valid,
            SUM(CASE WHEN "x_t+0_d1" NOT BETWEEN 0 AND 1 THEN 1 ELSE 0 END) AS d1_valid,
            SUM(CASE WHEN abs("x_t+0_d1" - (hour(t0) * 60 + minute(t0)) / 1440.0) > 0.0001 THEN 1 ELSE 0 END) AS waktu_konsisten,
            SUM(CASE WHEN "x_t+0_d0" = 0 THEN 1 ELSE 0 END) AS nol_tersisa
        FROM dasar
    ''').pl()


print('Pelanggaran pada data mentah:')
hitung_pelanggaran(f"read_parquet('{DATA_GLOB}')")

Pelanggaran pada data mentah:


total_baris,node_valid,waktu_valid,kecepatan_valid,d1_valid,waktu_konsisten,nol_tersisa
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
16930225,0,0,0,0,0,521


## 7. Simpan Data Bersih

Semua aturan di atas diterapkan, lalu hasilnya ditulis ke `data/processed/` lewat `sink_parquet`. Dua kolom turunan ditambahkan untuk memudahkan Milestone 3:
- `t0`: `t0_timestamp` yang sudah di-parse ke `Datetime`.
- `split`: asal berkas (`train`, `val`, `test`).

**Alasan:**
- `sink_parquet` menulis hasil secara streaming sehingga pembersihan 16,9 juta baris tidak membutuhkan RAM sebesar datanya.
- Hasil disimpan sebagai file terpisah di `data/processed/`, sedangkan `data/raw/` tidak diubah, sehingga proses dapat direproduksi dari data asli.
- Validasi diulang pada data bersih sebagai bukti bahwa seluruh pelanggaran sudah nol.

In [8]:
lf_bersih = (
    lf_dedup
    .filter(pl.all_horizontal([(pl.col(c).is_null()) | pl.col(c).is_between(0, KECEPATAN_MAKS) for c in KOLOM_D0]))
    .filter(pl.all_horizontal([pl.col(c).is_between(0, 1) for c in KOLOM_D1]))
    .with_columns(
        pl.col('t0_timestamp').str.to_datetime('%Y-%m-%dT%H:%M:%S').alias('t0'),
        pl.col('sumber').str.extract(r'([^/\\]+)\.parquet$', 1).alias('split'),
    )
    .drop('sumber')
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
lf_bersih.sink_parquet(OUTPUT_PATH)

hasil = pl.scan_parquet(OUTPUT_PATH)
n_bersih = hasil.select(pl.len()).collect(engine='streaming').item()

print(f'Baris sebelum cleaning: {jumlah_baris:,}')
print(f'Baris sesudah cleaning: {n_bersih:,}')
print(f'Baris terbuang        : {jumlah_baris - n_bersih:,} ({(jumlah_baris - n_bersih) / jumlah_baris * 100:.4f}%)')
print(f'Ukuran file bersih    : {OUTPUT_PATH.stat().st_size / 1024**2:.1f} MiB')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')
print()
print('Pelanggaran pada data bersih:')
hitung_pelanggaran(f"read_parquet('{OUTPUT_PATH.as_posix()}')")

Baris sebelum cleaning: 16,930,225
Baris sesudah cleaning: 16,930,225
Baris terbuang        : 0 (0.0000%)
Ukuran file bersih    : 755.0 MiB
Data bersih tersimpan di: D:\SEMESTER 7\BIG DATA\tugas1-abd-2026\data\processed\pems_bay_clean.parquet

Pelanggaran pada data bersih:


total_baris,node_valid,waktu_valid,kecepatan_valid,d1_valid,waktu_konsisten,nol_tersisa
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
16930225,0,0,0,0,0,0


## 8. Verifikasi Hasil Cleaning

Perbandingan sebelum dan sesudah cleaning pada kolom `x_t+0_d0`, serta jumlah baris per split.

In [9]:
perbandingan = pl.concat([
    lf.select(
        pl.lit('mentah').alias('tahap'),
        pl.col('x_t+0_d0').mean().round(3).alias('rata_rata'),
        pl.col('x_t+0_d0').min().alias('min'),
        pl.col('x_t+0_d0').null_count().alias('jumlah_null'),
    ).collect(engine='streaming'),
    hasil.select(
        pl.lit('bersih').alias('tahap'),
        pl.col('x_t+0_d0').mean().round(3).alias('rata_rata'),
        pl.col('x_t+0_d0').min().alias('min'),
        pl.col('x_t+0_d0').null_count().alias('jumlah_null'),
    ).collect(engine='streaming'),
])
print(perbandingan)
print()
print(hasil.group_by('split').agg(pl.len().alias('jumlah_baris'), pl.col('t0').min().alias('mulai'), pl.col('t0').max().alias('akhir'), pl.col('n_hilang').sum().alias('nilai_hilang')).sort('mulai').collect(engine='streaming'))

shape: (2, 4)
┌────────┬───────────┬─────┬─────────────┐
│ tahap  ┆ rata_rata ┆ min ┆ jumlah_null │
│ ---    ┆ ---       ┆ --- ┆ ---         │
│ str    ┆ f64       ┆ f64 ┆ u32         │
╞════════╪═══════════╪═════╪═════════════╡
│ mentah ┆ 62.617    ┆ 0.0 ┆ 0           │
│ bersih ┆ 62.619    ┆ 3.0 ┆ 521         │
└────────┴───────────┴─────┴─────────────┘



shape: (3, 5)
┌───────┬──────────────┬─────────────────────┬─────────────────────┬──────────────┐
│ split ┆ jumlah_baris ┆ mulai               ┆ akhir               ┆ nilai_hilang │
│ ---   ┆ ---          ┆ ---                 ┆ ---                 ┆ ---          │
│ str   ┆ u32          ┆ datetime[μs]        ┆ datetime[μs]        ┆ i64          │
╞═══════╪══════════════╪═════════════════════╪═════════════════════╪══════════════╡
│ train ┆ 11851125     ┆ 2017-01-01 00:55:00 ┆ 2017-05-07 16:35:00 ┆ 4128         │
│ val   ┆ 1692925      ┆ 2017-05-07 16:40:00 ┆ 2017-05-25 18:40:00 ┆ 3168         │
│ test  ┆ 3386175      ┆ 2017-05-25 18:45:00 ┆ 2017-06-30 22:55:00 ┆ 5208         │
└───────┴──────────────┴─────────────────────┴─────────────────────┴──────────────┘


## Catatan Milestone 2

1. Masalah kualitas utama adalah nilai hilang yang tersembunyi. Pemeriksaan awal menemukan `null` 0, duplikat kunci 0, kecepatan atau `d1` di luar batas 0, dan ketidakcocokan waktu 0. Kecepatan `0` (penanda sensor gagal baca) muncul 12.504 kali dari 406.325.400 nilai (±0,003%), tersebar pada 12.504 baris (±0,074%) dan merata antar sensor (paling banyak 4 kali per sensor pada `x_t+0_d0`).
2. Untuk missing values, nilai `0` diubah menjadi `null` dan kolom `n_hilang` ditambahkan per baris. Baris tidak dibuang dan tidak diimputasi, karena baris yang berdekatan berbagi nilai pada jendelanya dan nilai karangan akan membuat baris lain tidak konsisten. Jumlah `null` baru (12.504) sama dengan jumlah nol awal, dan pelanggaran `nol_tersisa` turun dari 521 menjadi 0.
3. Hanya ada satu celah waktu (1 jam 5 menit) pada 2017-03-12 03:00. Tanggal ini bertepatan dengan pergantian ke waktu musim panas AS (jam 02:00 lokal dilompati), jadi kemungkinan besar ini artefak zona waktu dan bukan data hilang. Celah tidak diisi. Operasi `shift()` dan `rolling_mean()` pada Milestone 3 bekerja per langkah dan tidak per jarak 5 menit.
4. Pelanggaran batas domain (0 sampai 100 mph dan `d1` 0 sampai 1) berjumlah 0. IQR per sensor pada `x_t+0_d0` menandai 1.572.424 nilai (9,29%) di bawah batas bawah dan 60.729 nilai (0,36%) di atas batas atas. Outlier tetap dipertahankan karena kecepatan rendah mencerminkan kemacetan nyata yang menjadi objek analisis.
5. Kunci (`node_id`, `t0_timestamp`) unik, duplikat penuh pada 50 kolom berjumlah 0, dan tidak ada timestamp yang muncul di lebih dari satu split. Pembagian `train`, `val`, dan `test` bebas dari kebocoran waktu.
6. Jumlah baris tetap 16.930.225 karena strategi menandai nilai hilang dan tidak menghapusnya. Data bersih tersimpan di `data/processed/pems_bay_clean.parquet` (±755 MiB) dengan dua kolom turunan, `t0` (Datetime) dan `split`. Rata-rata `x_t+0_d0` bergeser dari 62,617 ke 62,619 mph dan nilai minimumnya naik dari 0,0 ke 3,0 mph.
7. Enam aturan validasi SQL DuckDB (`node_valid`, `waktu_valid`, `kecepatan_valid`, `d1_valid`, `waktu_konsisten`, `nol_tersisa`) hanya mendeteksi pelanggaran `nol_tersisa` pada data mentah (521 baris). Data bersih lolos semua aturan dengan 0 pelanggaran.